# Phase 4 TrOCR Word-Level OCR and Final Full-Prescription Pipeline

This notebook trains/evaluates the word-level OCR model and runs the final full-prescription inference flow.

## Part A: Word-Level TrOCR Training and Evaluation

1. Load a word-level OCR dataset. The notebook auto-prefers `data/custom_word_ocr_dataset_augmented`, then `data/custom_word_ocr_dataset`, then the original BD dataset.
2. Fine-tune `microsoft/trocr-base-handwritten` for medicine-name recognition.
3. Evaluate exact-match accuracy, CER, and WER on unaugmented validation/test splits.
4. Save the best model and prediction CSV for thesis reporting.

## Part B: Final Full-Prescription Evaluation Pipeline

The final section runs:

1. preprocessing
2. YOLO handwritten-region detection
3. hybrid line segmentation
4. hybrid word segmentation
5. word-level TrOCR inference
6. drug lexicon matching plus dosage/frequency extraction
7. structured `predictions.csv` / `predictions.json`

Your OCR model is word-level, so final full-prescription inference must segment down to word crops before TrOCR.


In [ ]:
# Colab/Drive project setup.
# This cell mounts Google Drive, keeps the repository in Drive, and keeps data in Drive.
# Put raw prescriptions here after this cell runs:
# /content/drive/MyDrive/phase4_project/repo/data/raw

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    if not (REPO_DIR / 'pipeline').exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(
                f'{REPO_DIR} exists but does not look like the project repo. '
                'Move/rename it or set REPO_DIR to the correct folder.'
            )
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    # Local Jupyter/VS Code fallback: use the current repository folder.
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)

# data/ is gitignored, so it is safe to keep raw images and generated files here.
# In Colab this folder is inside Google Drive and persists across sessions.
DATA_DIR = REPO_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('Repository:', REPO_DIR)
print('Data folder:', DATA_DIR)
print('Current working directory:', Path.cwd())
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())
print('Raw data folder:', RAW_DIR)
print('Raw images currently:', len([p for p in RAW_DIR.glob('*') if p.is_file()]))


# Separate persistent folder for TrOCR checkpoints/results.
PROJECT_DIR = DRIVE_BASE / 'trocr_work' if IN_COLAB else REPO_DIR / 'trocr_work'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('TrOCR work folder:', PROJECT_DIR)

## Step 1: Install OCR Dependencies

Run this after the Drive/repository setup cell. These packages are required for TrOCR training and evaluation.


In [ ]:
!pip -q install -U transformers datasets evaluate jiwer accelerate sentencepiece kagglehub


## Imports


In [ ]:
import os
import random
import inspect
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    EarlyStoppingCallback,
)
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)


## Download Dataset from KaggleHub
Run this cell if dataset is not already available in Drive/Colab.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mamun1113/doctors-handwritten-prescription-bd-dataset")

print("Path to dataset files:", path)
KAGGLE_DATASET_PATH = Path(path)


## Config (T4 Fast Default)
If GPU memory is low, keep this fast setup.


In [ ]:
CFG = {
    'model_name': 'microsoft/trocr-base-handwritten',
    'target_col': 'MEDICINE_NAME',
    'image_col': 'IMAGE',
    'max_target_len': 24,
    'num_train_epochs': 8,
    'train_batch_size': 8,
    'eval_batch_size': 8,
    'grad_accum_steps': 1,
    'learning_rate': 3e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'num_beams': 1,
    'early_stopping_patience': 3,
    'output_dir': str(PROJECT_DIR / 'checkpoints'),
    'best_dir': str(PROJECT_DIR / 'best_model'),
    'pred_csv': str(PROJECT_DIR / 'phase4_test_predictions_trocr.csv'),
    'max_steps': 1200,
}
CFG


## Dataset Discovery

If auto-search fails, set `MANUAL_DATASET_BASE` to the folder containing `Training/Validation/Testing`. For the final thesis run, prefer `data/custom_word_ocr_dataset_augmented` because only its training split is augmented.


In [ ]:
MANUAL_DATASET_BASE = None

DATA_ROOT_HINTS = [
    str(PROJECT_DIR / 'data' / 'custom_word_ocr_dataset_augmented'),
    str(PROJECT_DIR / 'data' / 'custom_word_ocr_dataset'),
    str(DATA_DIR / 'custom_word_ocr_dataset_augmented'),
    str(DATA_DIR / 'custom_word_ocr_dataset'),
    '/content/drive/MyDrive/phase4_project/repo/data/custom_word_ocr_dataset_augmented',
    '/content/drive/MyDrive/phase4_project/repo/data/custom_word_ocr_dataset',
    '/content/data/doctors-handwritten-prescription-bd-dataset',
    '/content/drive/MyDrive/data/doctors-handwritten-prescription-bd-dataset',
    '/content/drive/MyDrive',
    str(PROJECT_DIR / 'data'),
    str(DATA_DIR),
    str(DATA_DIR / 'doctors-handwritten-prescription-bd-dataset'),
    str(PROJECT_DIR / 'data' / 'doctors-handwritten-prescription-bd-dataset'),
]
if 'KAGGLE_DATASET_PATH' in globals():
    DATA_ROOT_HINTS += [str(KAGGLE_DATASET_PATH), str(KAGGLE_DATASET_PATH.parent)]


def is_dataset_base(p: Path) -> bool:
    return (
        (p / 'Training' / 'training_labels.csv').exists()
        and (p / 'Validation' / 'validation_labels.csv').exists()
        and (p / 'Testing' / 'testing_labels.csv').exists()
    )


def find_dataset_base(hints):
    if MANUAL_DATASET_BASE:
        m = Path(MANUAL_DATASET_BASE)
        if is_dataset_base(m):
            return m
        print('MANUAL_DATASET_BASE invalid:', m)

    for hint in hints:
        p = Path(hint)
        if not p.exists():
            continue
        if is_dataset_base(p):
            return p
        for cand in p.rglob('*'):
            if cand.is_dir() and is_dataset_base(cand):
                return cand

    for f in glob('/content/**/training_labels.csv', recursive=True):
        cand = Path(f).parent.parent
        if is_dataset_base(cand):
            return cand
    return None

DATASET_BASE = find_dataset_base(DATA_ROOT_HINTS)
print('DATASET_BASE:', DATASET_BASE)
if DATASET_BASE is None:
    print('Debug candidates (training_labels.csv):')
    for f in glob('/content/**/training_labels.csv', recursive=True)[:30]:
        print(f)
    raise FileNotFoundError('Dataset base not found. Set MANUAL_DATASET_BASE.')


In [ ]:
SPLITS = {
    'train': {
        'csv': DATASET_BASE / 'Training' / 'training_labels.csv',
        'img_dir': DATASET_BASE / 'Training' / 'training_words',
    },
    'val': {
        'csv': DATASET_BASE / 'Validation' / 'validation_labels.csv',
        'img_dir': DATASET_BASE / 'Validation' / 'validation_words',
    },
    'test': {
        'csv': DATASET_BASE / 'Testing' / 'testing_labels.csv',
        'img_dir': DATASET_BASE / 'Testing' / 'testing_words',
    },
}

train_df = pd.read_csv(SPLITS['train']['csv'])
val_df = pd.read_csv(SPLITS['val']['csv'])
test_df = pd.read_csv(SPLITS['test']['csv'])

print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)
print('columns:', train_df.columns.tolist())
print('unique medicine names:', train_df[CFG['target_col']].nunique())
train_df.head(3)


In [ ]:
def count_missing(df, img_dir, image_col):
    img_dir = Path(img_dir)
    missing = [x for x in df[image_col].astype(str) if not (img_dir / x).exists()]
    return missing

for split_name, split_obj, df in [
    ('train', SPLITS['train'], train_df),
    ('val', SPLITS['val'], val_df),
    ('test', SPLITS['test'], test_df),
]:
    miss = count_missing(df, split_obj['img_dir'], CFG['image_col'])
    print(split_name, 'missing images:', len(miss))


In [ ]:
sample_df = train_df.sample(8, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    row = sample_df.iloc[i]
    img = Image.open(SPLITS['train']['img_dir'] / row[CFG['image_col']]).convert('RGB')
    ax.imshow(img)
    ax.set_title(str(row[CFG['target_col']]), fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Model + Processor
Uses updated `generation_config` to avoid recent `transformers` ValueError.


In [ ]:
processor = TrOCRProcessor.from_pretrained(CFG['model_name'])
model = VisionEncoderDecoderModel.from_pretrained(CFG['model_name'])

# token IDs stay in model.config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

# generation params must go to generation_config
model.generation_config.max_length = CFG['max_target_len']
model.generation_config.no_repeat_ngram_size = 0
model.generation_config.length_penalty = 1.0
model.generation_config.num_beams = CFG['num_beams']
# only valid for beam search (num_beams > 1)
model.generation_config.early_stopping = CFG['num_beams'] > 1

# speed-up on T4: freeze encoder
for p in model.encoder.parameters():
    p.requires_grad = False



In [ ]:
class PrescriptionWordOCRDataset(Dataset):
    def __init__(self, df, img_dir, processor, target_col, image_col='IMAGE', max_target_len=24):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.target_col = target_col
        self.image_col = image_col
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.img_dir / str(row[self.image_col])).convert('RGB')

        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        text = str(row[self.target_col]).strip()
        labels = self.processor.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_target_len,
            return_tensors='pt',
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}

train_ds = PrescriptionWordOCRDataset(train_df, SPLITS['train']['img_dir'], processor, CFG['target_col'], CFG['image_col'], CFG['max_target_len'])
val_ds = PrescriptionWordOCRDataset(val_df, SPLITS['val']['img_dir'], processor, CFG['target_col'], CFG['image_col'], CFG['max_target_len'])
test_ds = PrescriptionWordOCRDataset(test_df, SPLITS['test']['img_dir'], processor, CFG['target_col'], CFG['image_col'], CFG['max_target_len'])

print('dataset sizes:', len(train_ds), len(val_ds), len(test_ds))


In [ ]:
cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')

def norm_text(s):
    return ' '.join(str(s).strip().split())

def compute_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_texts = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_texts = [norm_text(x) for x in pred_texts]
    label_texts = [norm_text(x) for x in label_texts]

    exact = np.mean([p == y for p, y in zip(pred_texts, label_texts)])
    cer = cer_metric.compute(predictions=pred_texts, references=label_texts)
    wer = wer_metric.compute(predictions=pred_texts, references=label_texts)

    return {'exact_match': float(exact), 'cer': float(cer), 'wer': float(wer)}


In [ ]:
def build_training_args(cfg):
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    args_dict = {
        'output_dir': cfg['output_dir'],
        'predict_with_generate': True,
        'save_strategy': 'epoch',
        'logging_strategy': 'steps',
        'logging_steps': 50,
        'per_device_train_batch_size': cfg['train_batch_size'],
        'per_device_eval_batch_size': cfg['eval_batch_size'],
        'gradient_accumulation_steps': cfg['grad_accum_steps'],
        'num_train_epochs': cfg['num_train_epochs'],
        'learning_rate': cfg['learning_rate'],
        'warmup_ratio': cfg['warmup_ratio'],
        'weight_decay': cfg['weight_decay'],
        'save_total_limit': 2,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'exact_match',
        'greater_is_better': True,
        'fp16': torch.cuda.is_available(),
        'dataloader_num_workers': 2,
        'report_to': 'none',
        'remove_unused_columns': False,
        'seed': SEED,
        'max_steps': cfg['max_steps'],
    }
    if 'evaluation_strategy' in sig:
        args_dict['evaluation_strategy'] = 'epoch'
    elif 'eval_strategy' in sig:
        args_dict['eval_strategy'] = 'epoch'
    return Seq2SeqTrainingArguments(**args_dict)

training_args = build_training_args(CFG)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['early_stopping_patience'])],
)
training_args


In [ ]:
train_result = trainer.train()
print(train_result)


In [ ]:
val_metrics = trainer.evaluate(eval_dataset=val_ds, metric_key_prefix='val')
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
print('Validation:', val_metrics)
print('Test:', test_metrics)


In [ ]:
best_dir = Path(CFG['best_dir'])
best_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(best_dir))
processor.save_pretrained(str(best_dir))
print('Saved best model to:', best_dir)


In [ ]:
pred = trainer.predict(test_ds)
pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
pred_texts = [norm_text(x) for x in processor.batch_decode(pred_ids, skip_special_tokens=True)]

pred_df = test_df.copy()
pred_df['PREDICTED_MEDICINE_NAME'] = pred_texts
pred_df['IS_EXACT'] = (
    pred_df[CFG['target_col']].astype(str).map(norm_text) == pred_df['PREDICTED_MEDICINE_NAME']
).astype(int)

pred_csv = Path(CFG['pred_csv'])
pred_csv.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(pred_csv, index=False)
print('Saved predictions to:', pred_csv)
pred_df.head(15)


## Inference (Presentation-Time)
Load model from Drive and predict on a single crop or a folder of crops.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_saved_model(model_dir=None):
    if model_dir is None:
        model_dir = CFG['best_dir']
    proc = TrOCRProcessor.from_pretrained(model_dir)
    mdl = VisionEncoderDecoderModel.from_pretrained(model_dir).to(device)
    mdl.eval()
    return proc, mdl

def predict_single_crop(image_path, proc=None, mdl=None):
    if proc is None or mdl is None:
        proc, mdl = load_saved_model()

    image = Image.open(image_path).convert('RGB')
    pixel_values = proc(images=image, return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        gen_ids = mdl.generate(
            pixel_values,
            num_beams=CFG['num_beams'],
            max_length=CFG['max_target_len'],
            early_stopping=(CFG['num_beams'] > 1),
        )
    return norm_text(proc.batch_decode(gen_ids, skip_special_tokens=True)[0])

def predict_folder(folder_path, exts=('.png', '.jpg', '.jpeg', '.webp')):
    proc, mdl = load_saved_model()
    rows = []
    for p in sorted(Path(folder_path).iterdir()):
        if p.suffix.lower() in exts:
            rows.append({'file': p.name, 'prediction': predict_single_crop(p, proc, mdl)})
    return pd.DataFrame(rows)



In [ ]:
# Example inference
# print(predict_single_crop('/content/some_crop.png'))
# df_pred = predict_folder('/content/some_folder_with_crops')
# df_pred.head()


In [ ]:
# Optional: download artifacts from Colab runtime
# from google.colab import files
# files.download(CFG['pred_csv'])
# !zip -qr /content/phase4_best_model.zip "{CFG['best_dir']}"
# files.download('/content/phase4_best_model.zip')


## Final Evaluation: Full Prescription Pipeline

This section runs the final thesis pipeline:

1. preprocess full prescription pages
2. detect handwritten regions with `models/region_yolo_best.pt`
3. segment lines using the hybrid OpenCV line segmenter
4. segment word crops
5. run word-level TrOCR
6. apply lexicon matching and dosage/frequency extraction

Do not pass the weak YOLO line model here unless you are doing an ablation experiment.


In [ ]:
from pathlib import Path
import os

# Reuse the Drive-backed repo created in the setup cell.
os.chdir(REPO_DIR)
print('REPO_DIR:', REPO_DIR)
print('Has pipeline:', Path('pipeline').exists())
print('Raw prescription folder:', Path('data/raw').resolve())
print('Raw images:', len([p for p in Path('data/raw').glob('*') if p.is_file()]))

### Install Pipeline Dependencies
Use the core requirements for preprocessing, segmentation, and annotation packaging. Install the OCR requirements only when running TrOCR inference locally.


In [ ]:
# Core CV/data pipeline dependencies
# !python3 -m pip install -r pipeline/requirements.txt

# Optional: local TrOCR inference dependencies
# !python3 -m pip install -r pipeline/requirements-ocr.txt


### Step 1: Preprocess Full Prescription Images
Input folder: `data/raw`.
Output folder: `data/processed/pages`.


In [ ]:
!python3 pipeline/scripts/preprocess_pages.py \
  --input-dir data/raw \
  --output-dir data/processed/pages \
  --manifest-out data/processed/page_manifest.csv


### Step 2: Prepare Layout Annotation Package
This creates images/classes for CVAT or Label Studio. Annotate `header`, `handwritten_region`, and `footer`, then export YOLO labels into `data/processed/layout_yolo_labels`.


In [ ]:
!python3 pipeline/scripts/prepare_layout_annotation.py \
  --pages-dir data/processed/pages \
  --output-dir data/processed/layout_annotation_package \
  --copy-images

print('Next manual step: annotate package images in CVAT/Label Studio and export YOLO labels to data/processed/layout_yolo_labels')


### Step 3: Crop Handwritten Regions
Run after YOLO layout labels are available. These region crops isolate prescription handwriting from printed header/footer content.


In [ ]:
!python3 pipeline/scripts/crop_regions_from_yolo.py \
  --pages-dir data/processed/pages \
  --labels-dir data/processed/layout_yolo_labels \
  --class-map pipeline/config/layout_classes.txt \
  --target-label handwritten_region \
  --output-dir data/processed/regions \
  --manifest-out data/processed/region_manifest.csv


### Step 4: Segment Lines from Handwritten Regions
This creates line crops for each medication/instruction line and context images for review.


In [ ]:
!python3 pipeline/scripts/segment_lines.py \
  --region-manifest data/processed/region_manifest.csv \
  --output-dir data/processed/line_crops \
  --manifest-out data/processed/line_manifest.csv


### Step 5: Segment Word Crops for Word-Level Annotation
This produces word images similar to the previously used BD cropped-word dataset.


In [ ]:
!python3 pipeline/scripts/segment_words.py \
  --line-manifest data/processed/line_manifest.csv \
  --output-dir data/processed/word_crops \
  --manifest-out data/processed/word_manifest.csv


### Step 6: Create Word Annotation Sheet
Annotate `word_text`, `medicine_name`, `is_medicine`, and `review_status`. Use the Streamlit app locally for visual annotation.


In [ ]:
!python3 pipeline/scripts/create_word_annotation_manifest.py \
  --word-manifest data/processed/word_manifest.csv \
  --output-csv data/processed/word_annotations.csv

print('Local app command:')
print('streamlit run pipeline/app/word_annotator_app.py -- --manifest data/processed/word_manifest.csv --annotations data/processed/word_annotations.csv --annotator-id annotator_1')


### Step 7: Build Custom Word-Level OCR Dataset
After reviewing annotations, export medicine word crops to the same structure as the BD dataset:

- `Training/training_words/*.png`
- `Validation/validation_words/*.png`
- `Testing/testing_words/*.png`
- split label CSV files with `IMAGE,MEDICINE_NAME`


In [ ]:
!python3 pipeline/scripts/build_ocr_dataset.py \
  --annotations-csv data/processed/word_annotations.csv \
  --output-root data/custom_word_ocr_dataset \
  --image-path-column word_image_path \
  --label-column medicine_name \
  --approved-status reviewed \
  --seed 42

CUSTOM_DATASET_BASE = Path('data/custom_word_ocr_dataset')
print('CUSTOM_DATASET_BASE:', CUSTOM_DATASET_BASE)


### Step 8: Train/Fine-Tune TrOCR on the Custom Word Dataset

To reuse the earlier training cells, set `MANUAL_DATASET_BASE` to `data/custom_word_ocr_dataset_augmented` before the dataset discovery cell, then rerun the model/training/evaluation cells.


In [ ]:
# Use this before rerunning the earlier Dataset Discovery cell:
preferred_dataset = Path('data/custom_word_ocr_dataset_augmented')
if not preferred_dataset.exists():
    preferred_dataset = Path('data/custom_word_ocr_dataset')
MANUAL_DATASET_BASE = str(preferred_dataset.resolve())
print('MANUAL_DATASET_BASE =', MANUAL_DATASET_BASE)


### Step 9: Final End-to-End Inference

This command runs raw full prescriptions through preprocessing, trained region YOLO, hybrid line/word segmentation, TrOCR inference, and lexicon validation. Use your fine-tuned checkpoint path for `TROCR_MODEL`.


In [ ]:
# Replace this with your saved/fine-tuned checkpoint when available.
# Example Colab checkpoint: '/content/drive/MyDrive/mtech_phase4_trocr/best_model'
REGION_MODEL = Path('models/region_yolo_best.pt')
trained_checkpoint = Path(CFG['output_dir']) / 'best_model'
TROCR_MODEL = str(trained_checkpoint) if trained_checkpoint.exists() else 'microsoft/trocr-base-handwritten'

print('Region model:', REGION_MODEL.resolve(), REGION_MODEL.exists())
print('TrOCR model:', TROCR_MODEL)

!python3 pipeline/scripts/run_end_to_end.py \
  --input data/raw \
  --output-dir data/final_demo_trocr \
  --yolo-model "{REGION_MODEL}" \
  --target-class 0 \
  --ocr-backend trocr \
  --ocr-unit word \
  --trocr-model "{TROCR_MODEL}" \
  --line-padding 6 \
  --lexicon pipeline/config/drug_lexicon.txt


### Step 10: View Structured Output
The final output table contains OCR text, medicine lexicon match, dosage, frequency, and validation status.


In [ ]:
import pandas as pd
pred_path = Path('data/final_demo_trocr/predictions.csv')
if pred_path.exists():
    display(pd.read_csv(pred_path).head(20))
else:
    print('Predictions not found yet:', pred_path)


### Quick Post-OCR Validation Test
This pure text test is useful to show lexicon correction and dosage/frequency parsing even before running the image pipeline.


In [ ]:
!python3 pipeline/scripts/validate_prescription_text.py --text "Acita 650 mg BD"
